# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamzanPUCIT/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

---
Distributions first: earlier impressions and CTR are heavily right-skewed (use buckets/log), age ranges 90–564 days.

In [1]:
# Setup: load the starter dataset (works in a local clone and in Colab)
from pathlib import Path
import numpy as np
import pandas as pd

RAW_URL = ("https://raw.githubusercontent.com/RamzanPUCIT/flyrank-ml-internship/"
           "main/data/raw/content_refresh_anonymized.csv")
local = [p for p in [Path("../../data/raw/content_refresh_anonymized.csv"),
                     Path("data/raw/content_refresh_anonymized.csv")] if p.exists()]
df = pd.read_csv(local[0] if local else RAW_URL)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("rows x cols:", df.shape)

# My lane slice (same definition in every notebook)
MIN_PREV_IMPR = 100          # page needs real demand in the earlier window
lane = df[df["impressions_prev_30d"] >= MIN_PREV_IMPR].copy()
# proxy target: observed later-window outcome (last 30d vs the 30d before)
lane["demand_drop"] = (lane["impressions_last_30d"] < 0.8 * lane["impressions_prev_30d"]).astype(int)
lane["ctr_prev"] = lane.clicks_prev_30d / lane.impressions_prev_30d * 100
lane[["impressions_prev_30d", "ctr_prev", "content_age_days", "word_count"]].describe().round(2).T

rows x cols: (30000, 44)


,count,mean,std,min,25%,50%,75%,max
impressions_prev_30d,18010.0,2955.42,7718.36,100.0,293.0,797.00,2471.00,218786.00
ctr_prev,18010.0,0.25,0.38,0.0,0.0,0.11,0.35,6.78
content_age_days,18010.0,258.89,135.92,90.0,131.0,236.00,347.00,564.00
word_count,12785.0,3448.94,1498.70,692.0,2615.0,2962.00,3925.00,9546.00


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

---
Three signals vs drop rate (n shown): **CTR — CONFIRMED** (low CTR drops more), **content age — OPPOSITE** to 'old pages decay' (young pages drop more), **word count — MIXED** (small differences).

In [2]:
def bt(col, bins):
    return lane.groupby(pd.cut(lane[col], bins, include_lowest=True), observed=True).demand_drop.agg(n="size", drop_rate="mean").round(3)
display(bt("ctr_prev", [-0.01, 0.1, 0.5, 1, 100])); display(bt("content_age_days", [89, 180, 365, 5000])); display(bt("word_count", [0, 1500, 2500, 3500, 1e5]))

,n,drop_rate
ctr_prev,,
"(-0.011, 0.1]",8642,0.673
"(0.1, 0.5]",6399,0.591
"(0.5, 1.0]",2213,0.509
"(1.0, 100.0]",756,0.484


,n,drop_rate
content_age_days,,
"(88.999, 180.0]",7394,0.713
"(180.0, 365.0]",6404,0.613
"(365.0, 5000.0]",4212,0.450


,n,drop_rate
word_count,,
"(-0.001, 1500.0]",841,0.816
"(1500.0, 2500.0]",1531,0.679
"(2500.0, 3500.0]",6361,0.628
"(3500.0, 100000.0]",4052,0.686


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

---
Flag-linked test — **staleness** (behind the refresh flags): drop rate barely changes between recently-updated and older-updated pages → **MIXED**; the 181+ bucket is tiny. Not used in my rule.

In [3]:
bt("days_since_last_update", [0, 30, 90, 180, 400])

,n,drop_rate
days_since_last_update,,
"(-0.001, 30.0]",10900,0.608
"(30.0, 90.0]",85,0.588
"(90.0, 180.0]",7004,0.628
"(180.0, 400.0]",21,0.952


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

---
In practice: the queue should lean on earlier CTR and page youth, not on staleness or length. These verdicts are observed and directional on one snapshot.

In [4]:
{"ctr_prev": "CONFIRMED", "content_age_days": "OPPOSITE", "word_count": "MIXED", "staleness (flag)": "MIXED"}

{'ctr_prev': 'CONFIRMED',
 'content_age_days': 'OPPOSITE',
 'word_count': 'MIXED',
 'staleness (flag)': 'MIXED'}

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.